In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pickle
import matplotlib.pyplot as plt
from filterpy.kalman import UnscentedKalmanFilter, MerweScaledSigmaPoints

In [ ]:
# Recrear la arquitectura exacta del Surrogate Model
class SurrogateModel(nn.Module):
    def __init__(self, input_dim=14, output_dim=28, hidden_units=100, n_hidden=4):
        super(SurrogateModel, self).__init__()
        layers = []
        in_size = input_dim
        for _ in range(n_hidden):
            layers.append(nn.Linear(in_size, hidden_units))
            layers.append(nn.ReLU())
            in_size = hidden_units
        layers.append(nn.Linear(hidden_units, output_dim))
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

# Inicializar y cargar pesos
device = torch.device('cpu')
model = SurrogateModel()
ckpt = torch.load('surrogate_best.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Cargar scalers ajustados durante el entrenamiento
with open('surrogate_scalers.pkl', 'rb') as f:
    scaler_in, scaler_out = pickle.load(f)

In [ ]:
# Función de transición de estado (Random Walk)
def fx_random_walk(x, dt):
    # La matriz de transición F es la identidad
    return x

# Función de medición no lineal usando el modelo subrogado D(w, theta)
def hx_surrogate(theta_norm, w_norm):
    """
    Toma el estado oculto (theta) y las condiciones de vuelo (w) en espacio normalizado.
    Retorna la predicción de los sensores X_s en espacio normalizado.
    """
    # Concatenar [W(4) | Theta(10)]
    x_in = np.concatenate([w_norm, theta_norm])
    x_in_tensor = torch.tensor(x_in, dtype=torch.float32).unsqueeze(0)
    
    # Evaluar los puntos sigma en la red neuronal
    with torch.no_grad():
        y_pred = model(x_in_tensor).squeeze(0).numpy()
        
    # El surrogate predice 28 variables: [X_s(14) | X_v(14)]
    # El UKF solo se actualiza con los observables X_s
    return y_pred[:14]

In [ ]:
def run_ukf_for_engine(W_real, Xs_real, scaler_in, scaler_out):
    """
    W_real: Array de (T, 4) con el historial de vuelo
    Xs_real: Array de (T, 14) con los sensores reales registrados
    """
    T_steps = len(W_real)
    
    # 1. Preparar datos en espacio normalizado [-1, 1]
    # Rellenamos theta temporalmente con ceros para mantener la dimensión requerida por scaler_in
    dummy_theta = np.zeros((T_steps, 10))
    W_norm = scaler_in.transform(np.concatenate([W_real, dummy_theta], axis=1))[:, :4]
    
    # Para Xs_real, rellenamos Xv temporalmente con ceros para scaler_out
    dummy_xv = np.zeros((T_steps, 14))
    Xs_norm = scaler_out.transform(np.concatenate([Xs_real, dummy_xv], axis=1))[:, :14]

    # 2. Configurar Puntos Sigma (Algoritmo de Julier & Uhlmann)
    points = MerweScaledSigmaPoints(n=10, alpha=0.1, beta=2., kappa=0)
    
    # 3. Inicializar UKF
    ukf = UnscentedKalmanFilter(dim_x=10, dim_z=14, dt=1., 
                                fx=fx_random_walk, hx=hx_surrogate, 
                                points=points)
    
    # Estado inicial: Asumimos motor sano al inicio (theta inicializado en normalización de salud base)
    ukf.x = np.ones(10) * 0.95 # Valor ajustable dependiendo del extremo del scaler
    ukf.P = np.eye(10) * 0.1   # Covarianza de error inicial
    
    # Matrices de covarianza de ruido (Hiperparámetros fundamentales)
    ukf.Q = np.eye(10) * 1e-5  # Ruido de proceso (tasa de degradación)
    ukf.R = np.eye(14) * 1e-3  # Ruido de medición (incertidumbre de sensores)
    
    theta_estimated = []
    
    # 4. Loop de estimación temporal
    for t in range(T_steps):
        # Predicción a priori
        ukf.predict()
        
        # Actualización a posteriori pasando las condiciones de vuelo (W_t) como kwarg
        ukf.update(z=Xs_norm[t], w_norm=W_norm[t])
        
        theta_estimated.append(ukf.x.copy())
        
    return np.array(theta_estimated)

In [ ]:
# Aquí ingresas el tensor de una unidad real desde el dataset N-CMAPSS
# W_unit = ... 
# Xs_unit = ...

theta_hat_norm = run_ukf_for_engine(W_unit, Xs_unit, scaler_in, scaler_out)

plt.figure(figsize=(12, 6))
for i in range(10):
    plt.plot(theta_hat_norm[:, i], label=f'$\\theta_{{{i+1}}}$', alpha=0.8)

plt.title('Evolución Estimada de los Parámetros de Salud con Filtro de Kalman Unscented', fontsize=14)
plt.xlabel('Ciclos de Vuelo', fontsize=12)
plt.ylabel('Valor Normalizado', fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()